In [0]:
%run ./01-config

In [0]:
class Upserter:
    def __init__(self, merge_query, temp_view_name):
        self.merge_query = merge_query
        self.temp_view_name = temp_view_name
    
    def upsert(self, df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView(self.temp_view_name)
        spark.sql(self.merge_query)

In [0]:
class Gold():
    def __init__(self, env):
        self.config = Config()
        self.test_data_dir = self.config.base_dir_data + "/test_data"
        self.checkpoint_base = self.config.base_dir_checkpoint + "/checkpoints"
        self.catalog = env
        self.silver_db = self.config.silver_db
        self.gold_db = self.config.gold_db
        self.maxFilesPerTrigger= seld.config.maxFilesPerTrigger
        spark.sql(f"USE {self.catalog}.{self.gold_db}")
    
    def upsert_workout_bpm_summary(self, once=True, processing_time='15 seconds', startingVersion=0):
        from pyspark.sql import functions as F

        #idempotent - once a workout session is complete, it doesn't change state. Insert new records only.
        query = f"""
        MERGE INTO {self.catalog}.{self.db_name}.workout_bpm_summary a
        USING workout_bpm_summary_delta b
        ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id = b.session_id
        WHEN NOT MATCHED THEN INSERT *
        """

        data_upserter = Upserter(query, "workout_bpm_summary_delta")

        df_users = spark.read.table(f"{self.catalog}.{self.silver_db}.user_bins")
        
        df_delta = (spark.readStream
            .option("startingVersion", startingVersion)
            .table(f"{self.catalog}.{self.gold_db}.workout_bpm")
            .withWatermark("end_time", "30 seconds")
            .groupBy("user_id", "workout_id", "session_id", "end_time")
            .agg(F.min("heartrate").alias("min_bpm"), 
                 F.mean("heartrate").alias("avg_bpm"), 
                 F.max("heartrate").alias("max_bpm"),
                 F.count("heartrate").alias("num_recordings"))
            .join(df_users, ["user_id"])
            .select("workout_id", "session_id", "user_id", "age", "gender", "city", "state", "min_bpm", "avg_bpm", "max_bpm", "num_recordings")
        
        stream_writer = (
            df_delta.writeStream
            .foreachBatch(data_upserter.upsert())
            .outputMode("append")
            .option("checkpointLocation", f"{self.checkpoint_base}/workout_bpm_summary")
            .queryName("workout_bpm_summary_upsert_stream")
        )

        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "gold_p1")
    
    def upsert(self, once=True, processing_time="5 seconds"):
        import time
        start = int(time.time())
        print("\nExecuting gold layer upsert...")
        self.upsert_workout_bpm_summary(once, processing_time)
        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()
        print(f"Completed gold layer upsert in {int(time.time()) - start} seconds")

    def assert_count(self, table_name, expected_count, filter=true):
        print(f"Validating {table_name} count...", end='')
        actual_count = spark.read.table(f"{self.catalog}.{self.silver_db}.{table_name}").where(filter).count()
        assert actual_count == expected_count, f"Expected {expected_count} records in {table_name} but found {actual_count}"
        print("Found expected number of records: {expected_count} for table {table_name}")
    
    def assert_row(self, location, table_name, sets):
        print(f"Validatin records in {table_name}...", end='')
        expected_rows = spark.read.format("parquet").load(f"{self.test_data_dir}/{location}_{sets}.parquet").collect()
        actual_rows = spark.read.table(f"{self.catalog}.{self.gold_db}.{table_name}").collect()
        assert expected_rows == actual_rows, f"Expected {expected_rows} records in {table_name} but found {actual_rows}"
        print("Successful row validation in {table_name}")

    def validate(self, sets):
        import time
        start = int(time.time())
        print("\nValidating gold layer data...")
        self.assert_rows("7-gym_summary", "gym_summary", sets)
        if sets > 1:
            self.assert_rows("7-workout_bpm_summary", "workout_bpm_summary", sets)
        print(f"Validated gold layer data in {int(time.time()) - start} seconds")
        